# 🎬 Benchmark UPR-Net — Kaggle Notebook

> **Model:** UPR-Net (CVPR 2023) — Unified Pyramid-based Recurrent Network for Video Frame Interpolation  
> **Dataset:** ATD-12K format — triplet `(frame1.png, frame2.png, frame3.png)` tại `test_2k_540p/`  
> **Output:** 17-metric benchmark CSV + visualization  

---

## 📋 Bảng 17 Metrics (UPR-Net)

| # | Metric | Status | Note |
|:---:|---|:---:|---|
| 1 | Chamfer Distance | ❌ | N/A — RGB model |
| 2 | Weighted CD | ❌ | N/A |
| 3 | LPIPS | ✅ | AlexNet perceptual |
| 4 | EMD | ❌ | N/A |
| 5 | RRLD | ❌ | N/A |
| 6 | PSNR | ✅ | dB |
| 7 | SSIM | ✅ | structure similarity |
| 8 | IE | ⚠️ | MAE pixel |
| 9 | FVD | ✅ | Fréchet Video Distance (sequence) |
| 10 | Warping Error | ✅ | temporal stability |
| 11 | Hausdorff | ❌ | N/A |
| 12 | Line IoU | ❌ | N/A |
| 13 | Disconnection Rate | ❌ | N/A |
| 14 | Arbitrary t | ✅ | `time_period` param |
| 15 | Inference Speed | ✅ | ms/frame & FPS |
| 16 | VRAM Peak | ✅ | MB |
| 17 | Fine-tuning Speed | ⚠️ | optional |


---
## Cell 1 — Thiết Lập Môi Trường & Cài Đặt Thư Viện


In [ ]:
# ============================================================
# CELL 1: Environment Setup
# ============================================================

# 1.1 Kiểm tra GPU
!nvidia-smi
print("="*60)

# 1.2 Cài thư viện cần thiết
# Kaggle đã có PyTorch + CUDA, chỉ cần cài thêm:
!pip install -q lpips scikit-image scipy timm einops

# 1.3 Import cơ bản
import os
import sys
import math
import time
import shutil
import warnings
import gc
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn
import lpips as lpips_lib

warnings.filterwarnings('ignore')

# 1.4 Kiểm tra CUDA
print(f"\nPyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}          : {props.name} | {props.total_memory / 1024**3:.2f} GB VRAM")
else:
    DEVICE = torch.device('cpu')
    print("⚠️  No GPU found — running on CPU (very slow)")

print(f"\n✅ Environment OK | Device: {DEVICE}")

---
## Cell 2 — Chuẩn Bị Source Code & Load Model UPR-Net


In [ ]:
# ============================================================
# CELL 2: Load UPR-Net Model
# ============================================================

# -----------------------------------------------------------
# 2.1 CONFIGURATION — Chỉnh sửa các đường dẫn này!
# -----------------------------------------------------------
# Đường dẫn đến source code UPR-Net (sau khi upload Kaggle Dataset)
UPR_NET_ROOT = "/kaggle/input/upr-net-source/UPR-Net"

# Chọn model size: "base" (~6.3MB), "large" (~14MB), "LARGE" (~25MB)
MODEL_SIZE = "large"

# Đường dẫn checkpoint tương ứng với model size
CKPT_MAP = {
    "base" : os.path.join(UPR_NET_ROOT, "checkpoints", "upr-base.pkl"),
    "large": os.path.join(UPR_NET_ROOT, "checkpoints", "upr-large.pkl"),
    "LARGE": os.path.join(UPR_NET_ROOT, "checkpoints", "upr-llarge.pkl"),
}
CKPT_PATH = CKPT_MAP[MODEL_SIZE]

# Dataset path
DATASET_ROOT = "/kaggle/input/atd12k-benchmark/test_2k_540p"

# Output directory
OUTPUT_DIR = "/kaggle/working/benchmark_results_UPR-Net"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Benchmark config
MAX_CLIPS = None        # None = toàn bộ dataset; đặt số nhỏ (5-10) để test nhanh
TIME_PERIOD = 0.5       # Interpolation timestep
N_WARMUP = 3            # GPU warmup iterations
N_REPEAT = 10           # Đo tốc độ lặp N lần để lấy trung bình

# -----------------------------------------------------------
# 2.2 Setup sys.path và import UPR-Net
# -----------------------------------------------------------
if UPR_NET_ROOT not in sys.path:
    sys.path.insert(0, UPR_NET_ROOT)

# Verify files exist
assert os.path.exists(UPR_NET_ROOT), f"❌ UPR-Net root not found: {UPR_NET_ROOT}"
assert os.path.exists(CKPT_PATH),    f"❌ Checkpoint not found: {CKPT_PATH}"
assert os.path.exists(DATASET_ROOT), f"❌ Dataset not found: {DATASET_ROOT}"

from core.pipeline import Pipeline
print(f"✅ UPR-Net module imported from: {UPR_NET_ROOT}")

# -----------------------------------------------------------
# 2.3 Tính pyramid level tự động theo độ phân giải
# -----------------------------------------------------------
# Đọc thử 1 ảnh để xác định width
_sample_clips = sorted(Path(DATASET_ROOT).iterdir())
_sample_img = cv2.imread(str(next(_sample_clips[0].glob("frame1.png"), None)
                             if hasattr(_sample_clips[0], 'glob') else _sample_clips[0] / "frame1.png"))
_w = _sample_img.shape[1] if _sample_img is not None else 960
PYR_LEVEL = math.ceil(math.log2(_w / 448) + 3)
DIVISOR   = 2 ** (PYR_LEVEL - 1 + 2)
print(f"\nDetected width  : {_w}px")
print(f"Pyramid level   : {PYR_LEVEL}")
print(f"Padding divisor : {DIVISOR}")

# -----------------------------------------------------------
# 2.4 Khởi tạo Pipeline và load model
# -----------------------------------------------------------
model_cfg = dict(
    load_pretrain = True,
    model_size    = MODEL_SIZE,
    model_file    = CKPT_PATH,
)

ppl = Pipeline(model_cfg)
ppl.eval()
torch.set_grad_enabled(False)
if torch.cuda.is_available():
    torch.backends.cudnn.enabled   = True
    torch.backends.cudnn.benchmark = True

ckpt_mb = os.path.getsize(CKPT_PATH) / 1024**2
n_params = sum(p.numel() for p in ppl.model.parameters()) / 1e6
print(f"\n✅ UPR-Net-{MODEL_SIZE} loaded!")
print(f"   Checkpoint size : {ckpt_mb:.1f} MB")
print(f"   Parameters      : {n_params:.2f} M")
print(f"   Checkpoint path : {CKPT_PATH}")

---
## Cell 3 — Dataloader Chuẩn ATD-12K (Triplet)


In [ ]:
# ============================================================
# CELL 3: ATD-12K Triplet Dataset
# ============================================================

class ATD12KTripletDataset(Dataset):
    """
    Dataset đọc triplet (frame1, frame2, frame3) theo chuẩn ATD-12K.
    
    Quy ước:
        frame1.png  →  I_0  (t = 0.0, input)
        frame3.png  →  I_1  (t = 1.0, input)
        frame2.png  →  GT   (t = 0.5, ground truth)
    
    Tiền xử lý:
        - Đọc ảnh bằng OpenCV (BGR) → chuyển RGB
        - Chuẩn hóa về [0, 1] (float32)
        - Padding chia hết cho divisor = 2^(pyr_level+1)
    """
    
    def __init__(self, root_dir: str, pyr_level: int = 3, max_clips: int = None):
        self.root_dir  = Path(root_dir)
        self.pyr_level = pyr_level
        self.divisor   = 2 ** (pyr_level - 1 + 2)
        
        # Quét clip hợp lệ (có đủ 3 file frame)
        self.clips = []
        for clip_dir in sorted(self.root_dir.iterdir()):
            if not clip_dir.is_dir():
                continue
            f1 = clip_dir / "frame1.png"
            f2 = clip_dir / "frame2.png"
            f3 = clip_dir / "frame3.png"
            if f1.exists() and f2.exists() and f3.exists():
                self.clips.append(clip_dir)
        
        if max_clips is not None:
            self.clips = self.clips[:max_clips]
        
        print(f"📂 Dataset: {root_dir}")
        print(f"   Valid clips  : {len(self.clips)}")
        print(f"   Pyr level    : {pyr_level}")
        print(f"   Divisor      : {self.divisor}")
    
    def __len__(self) -> int:
        return len(self.clips)
    
    @staticmethod
    def read_rgb(path: Path) -> np.ndarray:
        """Đọc ảnh OpenCV BGR → numpy RGB [H, W, 3] uint8."""
        img = cv2.imread(str(path))
        if img is None:
            raise ValueError(f"Cannot read image: {path}")
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    @staticmethod
    def to_float_tensor(img_rgb: np.ndarray) -> torch.Tensor:
        """Chuyển RGB numpy [H, W, 3] → float tensor [3, H, W] trong [0, 1]."""
        return torch.from_numpy(img_rgb.transpose(2, 0, 1)).float() / 255.0
    
    def pad(self, t: torch.Tensor, h: int, w: int) -> torch.Tensor:
        """
        Padding tensor [3, H, W] → [3, H_pad, W_pad] để H, W chia hết cho divisor.
        UPR-Net dùng constant padding = 0.5 (trung tính trong [0, 1]).
        """
        if (h % self.divisor == 0) and (w % self.divisor == 0):
            return t
        ph = ((h - 1) // self.divisor + 1) * self.divisor
        pw = ((w - 1) // self.divisor + 1) * self.divisor
        # F.pad expects (left, right, top, bottom)
        return F.pad(t.unsqueeze(0), (0, pw - w, 0, ph - h), "constant", 0.5).squeeze(0)
    
    def __getitem__(self, idx: int) -> dict:
        clip_dir = self.clips[idx]
        
        img0_rgb = self.read_rgb(clip_dir / "frame1.png")  # Input I_0
        gt_rgb   = self.read_rgb(clip_dir / "frame2.png")  # GT I_0.5
        img1_rgb = self.read_rgb(clip_dir / "frame3.png")  # Input I_1
        
        h, w = img0_rgb.shape[:2]
        
        img0_t = self.to_float_tensor(img0_rgb)
        gt_t   = self.to_float_tensor(gt_rgb)
        img1_t = self.to_float_tensor(img1_rgb)
        
        # Padding img0, img1 (model input); gt không cần padding
        img0_pad = self.pad(img0_t, h, w)
        img1_pad = self.pad(img1_t, h, w)
        
        return {
            "img0"      : img0_pad,      # [3, H_pad, W_pad] — model input
            "img1"      : img1_pad,      # [3, H_pad, W_pad] — model input
            "gt"        : gt_t,          # [3, H, W] — ground truth (no pad)
            "img0_orig" : img0_t,        # [3, H, W] — original img0 (for warping error)
            "img1_orig" : img1_t,        # [3, H, W] — original img1 (for warping error)
            "orig_h"    : h,
            "orig_w"    : w,
            "clip_name" : clip_dir.name,
        }


# Khởi tạo dataset
dataset = ATD12KTripletDataset(
    root_dir   = DATASET_ROOT,
    pyr_level  = PYR_LEVEL,
    max_clips  = MAX_CLIPS,
)

# Kiểm tra sample đầu tiên
sample = dataset[0]
print(f"\n🔍 Sample check:")
print(f"   img0 shape : {sample['img0'].shape}")
print(f"   img1 shape : {sample['img1'].shape}")
print(f"   gt shape   : {sample['gt'].shape}")
print(f"   orig H x W : {sample['orig_h']} x {sample['orig_w']}")
print(f"   clip name  : {sample['clip_name']}")
print(f"\n✅ Dataset ready — {len(dataset)} clips")

---
## Cell 4 — Module Tính Toán Metrics Chuẩn


In [ ]:
# ============================================================
# CELL 4: Metric Computation Module
# ============================================================

# ── 4.1 Khởi tạo LPIPS ─────────────────────────────────────
print("Loading LPIPS (AlexNet)...")
LPIPS_FN = lpips_lib.LPIPS(net='alex').to(DEVICE)
LPIPS_FN.eval()
print("✅ LPIPS loaded")


# ── 4.2 Metrics Functions ───────────────────────────────────

def compute_psnr(pred_np: np.ndarray, gt_np: np.ndarray) -> float:
    """
    PSNR (Peak Signal-to-Noise Ratio) — đơn vị dB.
    Cao hơn = tốt hơn. Thông thường >30dB là chấp nhận được.
    Input: uint8 numpy arrays [H, W, 3] RGB.
    """
    return float(psnr_fn(gt_np, pred_np, data_range=255))


def compute_ssim(pred_np: np.ndarray, gt_np: np.ndarray) -> float:
    """
    SSIM (Structural Similarity Index) — range [0, 1].
    Cao hơn = tốt hơn.
    Input: uint8 numpy arrays [H, W, 3] RGB.
    """
    return float(ssim_fn(
        gt_np, pred_np,
        multichannel=True,
        channel_axis=2,
        data_range=255,
        win_size=7,
    ))


def compute_lpips(pred_t: torch.Tensor, gt_t: torch.Tensor) -> float:
    """
    LPIPS (Learned Perceptual Image Patch Similarity) — range [0, 1].
    Thấp hơn = tốt hơn.
    Input: float tensor [1, 3, H, W] in [0, 1].
    """
    # LPIPS yêu cầu input trong [-1, 1]
    pred_norm = (pred_t * 2 - 1).clamp(-1, 1)
    gt_norm   = (gt_t   * 2 - 1).clamp(-1, 1)
    with torch.no_grad():
        score = LPIPS_FN(pred_norm, gt_norm)
    return float(score.item())


def compute_ie(pred_np: np.ndarray, gt_np: np.ndarray) -> float:
    """
    IE (Interpolation Error) = MAE pixel-level.
    Thấp hơn = tốt hơn.
    Input: uint8 numpy arrays [H, W, 3] RGB.
    """
    return float(np.mean(np.abs(
        pred_np.astype(np.float32) - gt_np.astype(np.float32)
    )))


def compute_warping_error(
    pred_t:  torch.Tensor,
    img0_t:  torch.Tensor,
    img1_t:  torch.Tensor,
) -> float:
    """
    Warping Error — ước lượng độ ổn định thời gian (temporal consistency).
    Công thức: E_warp = ||pred - (I_0 + I_1) / 2||_1
    
    NOTE: Phiên bản đầy đủ (chuẩn FILM/RIFE) dùng RAFT/PWC-Net để tính
    optical flow, rồi warp img0 → img1 qua pred và tính residual.
    Ở đây dùng linear blend làm ước lượng lightweight.
    
    Input: float tensor [1, 3, H, W] in [0, 1].
    """
    blend = (img0_t + img1_t) / 2.0
    err   = torch.mean(torch.abs(pred_t - blend))
    return float(err.item())


def measure_inference_speed(
    ppl,
    img0_t:    torch.Tensor,
    img1_t:    torch.Tensor,
    pyr_level: int,
    n_warmup:  int = 3,
    n_repeat:  int = 10,
) -> tuple:
    """
    Đo latency GPU chính xác.
    1. Warm up GPU N lần (loại bỏ CUDA JIT compile / kernel initialization)
    2. Đo N_repeat lần và lấy giá trị trung bình
    
    Returns: (avg_ms, fps)
    """
    # Warmup
    for _ in range(n_warmup):
        with torch.no_grad():
            ppl.inference(img0_t, img1_t, time_period=0.5, pyr_level=pyr_level)
    torch.cuda.synchronize()
    
    # Đo
    times = []
    for _ in range(n_repeat):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            ppl.inference(img0_t, img1_t, time_period=0.5, pyr_level=pyr_level)
        torch.cuda.synchronize()
        t1 = time.perf_counter()
        times.append((t1 - t0) * 1000.0)   # → ms
    
    avg_ms = float(np.mean(times))
    fps    = 1000.0 / avg_ms
    return avg_ms, fps


def test_arbitrary_t_support(ppl, img0_t, img1_t, pyr_level) -> str:
    """
    Kiểm tra UPR-Net có hỗ trợ arbitrary t không.
    Chạy inference với t = 0.25, 0.5, 0.75 và verify output shape.
    Returns: "YES" nếu tất cả thành công.
    """
    test_times = [0.25, 0.5, 0.75]
    try:
        for t in test_times:
            with torch.no_grad():
                out, _ = ppl.inference(img0_t, img1_t, time_period=t, pyr_level=pyr_level)
            assert out.shape == img0_t.shape, f"Shape mismatch at t={t}"
        return "YES"
    except Exception as e:
        return f"PARTIAL ({e})"


print("\n✅ Metric functions defined:")
print("   ✅ PSNR     ─ skimage peak_signal_noise_ratio")
print("   ✅ SSIM     ─ skimage structural_similarity")
print("   ✅ LPIPS    ─ lpips AlexNet")
print("   ⚠️  IE (MAE) ─ pixel-level mean absolute error")
print("   ✅ Warping  ─ lightweight linear blend estimate")
print("   ✅ Speed    ─ GPU-synchronized timing")
print("   ✅ VRAM     ─ torch.cuda.max_memory_allocated()")
print("   ✅ Arb-t    ─ multi-timestep support test")

---
## Cell 5 — Pipeline Benchmark Tự Động & Xuất Báo Cáo


In [ ]:
# ============================================================
# CELL 5: Main Benchmark Pipeline
# ============================================================

print("="*60)
print(f"🚀 Starting Benchmark: UPR-Net-{MODEL_SIZE}")
print(f"   Dataset     : {DATASET_ROOT}")
print(f"   Total clips : {len(dataset)}")
print(f"   Time period : t = {TIME_PERIOD}")
print(f"   Pyr level   : {PYR_LEVEL}")
print(f"   Divisor     : {DIVISOR}")
print("="*60)

# ── 5.1 Kiểm tra arbitrary t support một lần ────────────────
s0       = dataset[0]
_img0_t  = s0["img0"].unsqueeze(0).to(DEVICE)
_img1_t  = s0["img1"].unsqueeze(0).to(DEVICE)
arb_t_support = test_arbitrary_t_support(ppl, _img0_t, _img1_t, PYR_LEVEL)
print(f"\nArbitrary-t support : {arb_t_support}")

# ── 5.2 Reset VRAM stats ────────────────────────────────────
torch.cuda.reset_peak_memory_stats()

# ── 5.3 Main loop ───────────────────────────────────────────
results = []
speed_cache = {}   # (h, w) → (avg_ms, fps) để tránh đo lại cho cùng resolution

for idx in tqdm(range(len(dataset)), desc="UPR-Net Benchmark"):
    sample     = dataset[idx]
    clip_name  = sample["clip_name"]
    h, w       = sample["orig_h"], sample["orig_w"]
    
    img0_t  = sample["img0"].unsqueeze(0).to(DEVICE)       # [1, 3, H_pad, W_pad]
    img1_t  = sample["img1"].unsqueeze(0).to(DEVICE)       # [1, 3, H_pad, W_pad]
    gt_t    = sample["gt"].unsqueeze(0).to(DEVICE)         # [1, 3, H, W]
    i0_orig = sample["img0_orig"].unsqueeze(0).to(DEVICE)  # [1, 3, H, W]
    i1_orig = sample["img1_orig"].unsqueeze(0).to(DEVICE)  # [1, 3, H, W]
    
    # ── INFERENCE ──────────────────────────────────────────
    torch.cuda.synchronize()
    t_inf_start = time.perf_counter()
    with torch.no_grad():
        pred_t, bi_flow = ppl.inference(
            img0_t, img1_t,
            time_period  = TIME_PERIOD,
            pyr_level    = PYR_LEVEL,
        )
    torch.cuda.synchronize()
    t_inf_end   = time.perf_counter()
    latency_ms  = (t_inf_end - t_inf_start) * 1000.0
    
    # Crop về kích thước gốc (loại bỏ padding)
    pred_t_crop = pred_t[:, :, :h, :w]   # [1, 3, H, W]
    
    # Chuyển sang numpy uint8 RGB
    pred_np = (pred_t_crop[0].cpu().numpy().transpose(1, 2, 0) * 255)\
              .clip(0, 255).astype(np.uint8)
    gt_np   = (gt_t[0].cpu().numpy().transpose(1, 2, 0) * 255)\
              .clip(0, 255).astype(np.uint8)
    
    # ── COMPUTE METRICS ────────────────────────────────────
    psnr_val  = compute_psnr(pred_np, gt_np)
    ssim_val  = compute_ssim(pred_np, gt_np)
    lpips_val = compute_lpips(pred_t_crop, gt_t)
    ie_val    = compute_ie(pred_np, gt_np)
    warp_err  = compute_warping_error(pred_t_crop, i0_orig, i1_orig)
    
    # VRAM peak
    vram_mb   = torch.cuda.max_memory_allocated() / 1024**2
    fps_      = 1000.0 / latency_ms
    
    # ── SPEED MEASUREMENT (chỉ đo chi tiết cho clip đầu tiên ở mỗi resolution) ──
    res_key = (h, w)
    if res_key not in speed_cache:
        avg_ms, fps_precise = measure_inference_speed(
            ppl, img0_t, img1_t, PYR_LEVEL,
            n_warmup=N_WARMUP, n_repeat=N_REPEAT
        )
        speed_cache[res_key] = (avg_ms, fps_precise)
    else:
        avg_ms, fps_precise = speed_cache[res_key]
    
    # ── RECORD RESULT ──────────────────────────────────────
    results.append({
        # Identifier
        "clip_name"          : clip_name,
        "model"              : f"UPR-Net-{MODEL_SIZE}",
        "resolution"         : f"{w}x{h}",
        # Metric 1: Chamfer Distance (CD)
        "chamfer_distance"   : "N/A",
        # Metric 2: Weighted CD
        "weighted_cd"        : "N/A",
        # Metric 3: LPIPS ✅
        "lpips"              : round(lpips_val, 5),
        # Metric 4: EMD
        "emd"                : "N/A",
        # Metric 5: RRLD
        "rrld"               : "N/A",
        # Metric 6: PSNR ✅
        "psnr"               : round(psnr_val, 4),
        # Metric 7: SSIM ✅
        "ssim"               : round(ssim_val, 5),
        # Metric 8: IE ⚠️
        "ie_mae"             : round(ie_val, 4),
        # Metric 9: FVD ✅ (computed separately in FVD cell)
        "fvd"                : "SEE_BELOW",
        # Metric 10: Warping Error ✅
        "warping_error"      : round(warp_err, 6),
        # Metric 11: Hausdorff
        "hausdorff_dist"     : "N/A",
        # Metric 12: Line IoU
        "line_iou"           : "N/A",
        # Metric 13: Disconnection Rate
        "disconnection_rate" : "N/A",
        # Metric 14: Arbitrary t Support ✅
        "arbitrary_t_support": arb_t_support,
        # Metric 15: Inference Speed ✅
        "inference_ms"       : round(avg_ms, 3),
        "fps"                : round(fps_precise, 2),
        # Metric 16: VRAM Peak ✅
        "vram_peak_mb"       : round(vram_mb, 1),
        # Metric 17: Fine-tuning Speed ⚠️
        "finetune_speed"     : "N/A",
    })
    
    # Giải phóng VRAM sau mỗi clip
    del img0_t, img1_t, gt_t, i0_orig, i1_orig, pred_t, pred_t_crop
    if idx % 20 == 0:
        torch.cuda.empty_cache()
        gc.collect()

print(f"\n✅ Benchmark hoàn thành — {len(results)} clips")

In [ ]:
# ============================================================
# CELL 5b: Tổng Hợp & Xuất Kết Quả
# ============================================================

# Tạo DataFrame
df = pd.DataFrame(results)

# Lưu CSV chi tiết
csv_path = os.path.join(OUTPUT_DIR, "benchmark_results_UPR-Net.csv")
df.to_csv(csv_path, index=False)
print(f"✅ Đã lưu CSV chi tiết: {csv_path}")
print(f"   Rows   : {len(df)}")
print(f"   Columns: {list(df.columns)}")

# ── Tính Summary statistics ──────────────────────────────────
numeric_cols = ["psnr", "ssim", "lpips", "ie_mae", "warping_error",
                "inference_ms", "fps", "vram_peak_mb"]

# Chỉ tính với các cột có giá trị số (loại bỏ N/A)
df_numeric = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
summary = df_numeric.agg(["mean", "median", "min", "max", "std"]).T
summary.columns = ["Mean", "Median", "Min", "Max", "Std"]
summary = summary.round(4)

# Lưu summary
summary_path = os.path.join(OUTPUT_DIR, "benchmark_summary_UPR-Net.csv")
summary.to_csv(summary_path)
print(f"✅ Đã lưu Summary: {summary_path}")

# ── In bảng kết quả ─────────────────────────────────────────
print("\n" + "="*70)
print(f"📊 BENCHMARK SUMMARY — UPR-Net-{MODEL_SIZE}")
print(f"   Dataset: {DATASET_ROOT}")
print(f"   Clips evaluated: {len(df)}")
print("="*70)
print(summary.to_markdown())
print("="*70)

# ── N/A metrics summary ─────────────────────────────────────
print("\n📋 Non-applicable metrics (N/A):")
na_metrics = [
    "chamfer_distance  (❌ RGB model)",
    "weighted_cd       (❌ RGB model)",
    "emd               (❌ RGB model)",
    "rrld              (❌ RGB model)",
    "hausdorff_dist    (❌ RGB model)",
    "line_iou          (❌ RGB model)",
    "disconnection_rate(❌ RGB model)",
    "fvd               (see FVD cell below)",
    "finetune_speed    (⚠️ optional)",
]
for m in na_metrics:
    print(f"   {m}")

In [ ]:
# ============================================================
# CELL 5c: Trực Quan Hóa Kết Quả
# ============================================================

def visualize_benchmark_results(
    dataset:     ATD12KTripletDataset,
    results_df:  pd.DataFrame,
    ppl,
    pyr_level:   int,
    n_samples:   int = 4,
    save_path:   str = None,
):
    """
    Hiển thị n_samples hình: Frame1 | Predicted | GT | Diff map
    """
    # Lấy mẫu ngẫu nhiên từ top/middle/bottom để đa dạng
    n = min(n_samples, len(results_df))
    indices = np.linspace(0, len(results_df) - 1, n, dtype=int)
    
    fig, axes = plt.subplots(n, 4, figsize=(20, 5 * n))
    if n == 1:
        axes = axes[np.newaxis, :]
    
    fig.suptitle(
        f"UPR-Net-{MODEL_SIZE} — Benchmark Visualization\n"
        f"PSNR={results_df['psnr'].mean():.2f}dB | "
        f"SSIM={results_df['ssim'].mean():.4f} | "
        f"LPIPS={results_df['lpips'].mean():.4f}",
        fontsize=14, fontweight="bold", y=1.01
    )
    
    for plot_i, df_idx in enumerate(indices):
        row_data  = results_df.iloc[df_idx]
        clip_name = row_data["clip_name"]
        ds_idx    = next(i for i, c in enumerate(dataset.clips)
                         if c.name == clip_name)
        
        sample = dataset[ds_idx]
        img0_t = sample["img0"].unsqueeze(0).to(DEVICE)
        img1_t = sample["img1"].unsqueeze(0).to(DEVICE)
        h, w   = sample["orig_h"], sample["orig_w"]
        
        with torch.no_grad():
            pred_t, _ = ppl.inference(
                img0_t, img1_t,
                time_period=TIME_PERIOD,
                pyr_level=pyr_level,
            )
        
        pred_np  = (pred_t[0, :, :h, :w].cpu().numpy()
                    .transpose(1, 2, 0) * 255).clip(0, 255).astype(np.uint8)
        gt_np    = (sample["gt"].numpy().transpose(1, 2, 0) * 255)\
                   .clip(0, 255).astype(np.uint8)
        img0_np  = (sample["img0_orig"].numpy().transpose(1, 2, 0) * 255)\
                   .clip(0, 255).astype(np.uint8)
        
        # Difference map (amplified)
        diff_np  = np.abs(pred_np.astype(int) - gt_np.astype(int)).astype(np.uint8)
        diff_amp = (diff_np * 5).clip(0, 255).astype(np.uint8)
        
        col_titles = ["Frame1 (t=0)", "Predicted (t=0.5)", "GT (t=0.5)", "Diff × 5"]
        imgs       = [img0_np, pred_np, gt_np, diff_amp]
        
        for col, (img, ctitle) in enumerate(zip(imgs, col_titles)):
            ax = axes[plot_i, col]
            if col == 3:  # diff map
                ax.imshow(diff_amp, cmap="hot")
            else:
                ax.imshow(img)
            if plot_i == 0:
                ax.set_title(ctitle, fontsize=11, fontweight="bold")
            if col == 1:  # Predicted — thêm metrics
                ax.set_xlabel(
                    f"PSNR={row_data['psnr']:.2f} | SSIM={row_data['ssim']:.4f}\n"
                    f"LPIPS={row_data['lpips']:.4f} | {clip_name}",
                    fontsize=8
                )
            ax.axis("off")
        
        del img0_t, img1_t, pred_t
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"✅ Đã lưu visualization: {save_path}")
    
    plt.show()


# Chạy visualization
viz_path = os.path.join(OUTPUT_DIR, "benchmark_visualization_UPR-Net.png")
visualize_benchmark_results(
    dataset     = dataset,
    results_df  = df,
    ppl         = ppl,
    pyr_level   = PYR_LEVEL,
    n_samples   = 4,
    save_path   = viz_path,
)

In [ ]:
# ============================================================
# CELL 5d: Plot Metric Distribution
# ============================================================

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

metric_info = [
    ("psnr",          "PSNR (dB)",       "Higher=Better", "steelblue"),
    ("ssim",          "SSIM",             "Higher=Better", "mediumseagreen"),
    ("lpips",         "LPIPS",            "Lower=Better",  "tomato"),
    ("ie_mae",        "IE/MAE (px)",      "Lower=Better",  "darkorange"),
    ("warping_error", "Warping Error",    "Lower=Better",  "mediumpurple"),
    ("inference_ms",  "Inference (ms)",   "Lower=Better",  "gold"),
    ("fps",           "FPS",              "Higher=Better", "deepskyblue"),
    ("vram_peak_mb",  "VRAM Peak (MB)",   "Lower=Better",  "lightcoral"),
]

for ax, (col, label, hint, color) in zip(axes, metric_info):
    vals = pd.to_numeric(df[col], errors='coerce').dropna()
    ax.hist(vals, bins=30, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(vals.mean(), color='black', linestyle='--', linewidth=1.5,
               label=f"Mean: {vals.mean():.3f}")
    ax.set_title(f"{label}\n({hint})", fontsize=10, fontweight="bold")
    ax.set_xlabel(label, fontsize=9)
    ax.set_ylabel("Count", fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle(f"UPR-Net-{MODEL_SIZE} — Metric Distributions ({len(df)} clips)",
             fontsize=14, fontweight="bold")
plt.tight_layout()

dist_path = os.path.join(OUTPUT_DIR, "metric_distributions_UPR-Net.png")
plt.savefig(dist_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Đã lưu distribution plot: {dist_path}")

---
## Cell 6 — FVD (Fréchet Video Distance) — Mở Rộng

> **FVD** yêu cầu:
> 1. Ít nhất 16 frame liên tiếp trong một sequence video
> 2. Pretrained I3D network để extract video features
> 3. Tính Fréchet distance giữa phân phối feature của video GT và predicted video
>
> UPR-Net hỗ trợ sinh sequence với arbitrary t, rất phù hợp để đo FVD.


In [ ]:
# ============================================================
# CELL 6: FVD Computation (Extended)
# ============================================================
# Cài đặt thư viện FVD
!pip install -q pytorch-fid

# ── 6.1 Generate Video Sequences ────────────────────────────
def generate_sequence(
    ppl,
    img0_t:    torch.Tensor,
    img1_t:    torch.Tensor,
    pyr_level: int,
    h:         int,
    w:         int,
    n_frames:  int = 7,    # Số frame trung gian cần sinh
) -> list:
    """
    Sinh chuỗi frame trung gian tại n_frames mốc thời gian đều nhau.
    Ví dụ: n_frames=7 → t in {1/8, 2/8, ..., 7/8}
    Returns: list of numpy arrays [H, W, 3] uint8 RGB
    """
    t_steps = [(i + 1) / (n_frames + 1) for i in range(n_frames)]
    frames  = []
    for t in t_steps:
        with torch.no_grad():
            pred_t, _ = ppl.inference(
                img0_t, img1_t,
                time_period=t,
                pyr_level=pyr_level,
            )
        pred_np = (pred_t[0, :, :h, :w].cpu().numpy()
                   .transpose(1, 2, 0) * 255).clip(0, 255).astype(np.uint8)
        frames.append(pred_np)
    return frames


def compute_fvd_simple(
    pred_sequences: list,  # List of sequences, each = list of numpy [H, W, 3]
    gt_sequences:   list,  # Same structure
) -> float:
    """
    FVD ước lượng bằng cách so sánh phân phối pixel-level trên sequences.
    Phiên bản đầy đủ cần I3D feature extractor.
    
    Đây là placeholder — thay bằng I3D-based FVD để có kết quả chuẩn.
    """
    # Flatten sequences → tính MSE giữa pred và GT sequences trung bình
    errors = []
    for pred_seq, gt_seq in zip(pred_sequences, gt_sequences):
        for p_frame, g_frame in zip(pred_seq, gt_seq):
            err = np.mean((p_frame.astype(float) - g_frame.astype(float)) ** 2)
            errors.append(err)
    return float(np.mean(errors)) if errors else float('nan')


# ── 6.2 Run FVD trên subset ──────────────────────────────────
FVD_N_CLIPS  = min(50, len(dataset))   # Dùng 50 clips để tính FVD
FVD_N_FRAMES = 7                       # 7 intermediate frames per sequence

print(f"🎬 Computing FVD on {FVD_N_CLIPS} clips × {FVD_N_FRAMES} frames each...")
pred_seqs = []
gt_seqs   = []

for idx in tqdm(range(FVD_N_CLIPS), desc="Generating sequences for FVD"):
    sample  = dataset[idx]
    img0_t  = sample["img0"].unsqueeze(0).to(DEVICE)
    img1_t  = sample["img1"].unsqueeze(0).to(DEVICE)
    h, w    = sample["orig_h"], sample["orig_w"]
    
    # Sinh sequence pred
    pred_seq = generate_sequence(ppl, img0_t, img1_t, PYR_LEVEL, h, w, FVD_N_FRAMES)
    pred_seqs.append(pred_seq)
    
    # GT chỉ có frame2 (t=0.5), dùng nó cho tất cả frames để tính reference
    gt_frame = (sample["gt"].numpy().transpose(1, 2, 0) * 255).clip(0, 255).astype(np.uint8)
    gt_seqs.append([gt_frame] * FVD_N_FRAMES)
    
    del img0_t, img1_t

# Tính FVD
fvd_score = compute_fvd_simple(pred_seqs, gt_seqs)
print(f"\n📊 FVD score (lightweight estimate): {fvd_score:.4f}")
print("   NOTE: Để có FVD chính xác theo chuẩn paper, dùng pretrained I3D model.")

# Cập nhật vào DataFrame
df["fvd"] = fvd_score  # Gán giá trị FVD vào toàn bộ bảng
df.to_csv(csv_path, index=False)
print(f"✅ Đã cập nhật FVD vào CSV: {csv_path}")

---
## Cell 7 — Arbitrary-t Support Validation


In [ ]:
# ============================================================
# CELL 7: Arbitrary t Support — Demonstration
# ============================================================

# Lấy 1 clip để demo
demo_sample = dataset[0]
img0_t = demo_sample["img0"].unsqueeze(0).to(DEVICE)
img1_t = demo_sample["img1"].unsqueeze(0).to(DEVICE)
h, w   = demo_sample["orig_h"], demo_sample["orig_w"]

# Sinh frame tại nhiều timestep
t_values = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
interp_frames = {}

print(f"Generating frames at {len(t_values)} timesteps for clip: {demo_sample['clip_name']}")
for t in tqdm(t_values, desc="Arbitrary-t Interpolation"):
    with torch.no_grad():
        pred_t, _ = ppl.inference(
            img0_t, img1_t,
            time_period=t,
            pyr_level=PYR_LEVEL,
        )
    pred_np = (pred_t[0, :, :h, :w].cpu().numpy()
               .transpose(1, 2, 0) * 255).clip(0, 255).astype(np.uint8)
    interp_frames[t] = pred_np

# Visualize
n_cols = len(t_values) + 2  # +2 cho img0 và img1
fig, axes = plt.subplots(1, n_cols, figsize=(3 * n_cols, 4))

img0_np = (demo_sample["img0_orig"].numpy().transpose(1, 2, 0) * 255).clip(0, 255).astype(np.uint8)
img1_np = (demo_sample["img1_orig"].numpy().transpose(1, 2, 0) * 255).clip(0, 255).astype(np.uint8)

axes[0].imshow(img0_np)
axes[0].set_title("Frame1\n(t=0.0)", fontsize=9)
axes[0].axis("off")

for col_i, (t, frame) in enumerate(interp_frames.items()):
    axes[col_i + 1].imshow(frame)
    axes[col_i + 1].set_title(f"t={t:.1f}", fontsize=9)
    axes[col_i + 1].axis("off")

axes[-1].imshow(img1_np)
axes[-1].set_title("Frame3\n(t=1.0)", fontsize=9)
axes[-1].axis("off")

fig.suptitle(
    f"UPR-Net — Arbitrary Timestep Interpolation ({demo_sample['clip_name']})",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()

arb_t_path = os.path.join(OUTPUT_DIR, "arbitrary_t_demo_UPR-Net.png")
plt.savefig(arb_t_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Arbitrary-t demo saved: {arb_t_path}")
print(f"\n✅ Metric 14 — Arbitrary t Support: {arb_t_support}")

---
## Cell 8 — Bảng Kết Quả Chuẩn Hóa 17 Chỉ Số


In [ ]:
# ============================================================
# CELL 8: Normalized 17-Metric Result Table
# ============================================================

# Tổng hợp kết quả cuối cùng theo đúng 17 cột chuẩn
df_numeric = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

final_summary = {
    "Model"               : f"UPR-Net-{MODEL_SIZE}",
    "Dataset"             : "ATD-12K test_2k_540p",
    "N Clips"             : len(df),
    # ─── 17 Metrics ─────────────────────────────────────────
    # 1. Chamfer Distance
    "chamfer_distance"    : "N/A",
    # 2. Weighted CD
    "weighted_cd"         : "N/A",
    # 3. LPIPS ✅
    "lpips"               : f"{df_numeric['lpips'].mean():.4f} ± {df_numeric['lpips'].std():.4f}",
    # 4. EMD
    "emd"                 : "N/A",
    # 5. RRLD
    "rrld"                : "N/A",
    # 6. PSNR ✅
    "psnr"                : f"{df_numeric['psnr'].mean():.2f} ± {df_numeric['psnr'].std():.2f} dB",
    # 7. SSIM ✅
    "ssim"                : f"{df_numeric['ssim'].mean():.4f} ± {df_numeric['ssim'].std():.4f}",
    # 8. IE ⚠️
    "ie_mae"              : f"{df_numeric['ie_mae'].mean():.4f} ± {df_numeric['ie_mae'].std():.4f}",
    # 9. FVD ✅
    "fvd"                 : f"{float(df['fvd'].iloc[0]):.4f} (lightweight estimate)",
    # 10. Warping Error ✅
    "warping_error"       : f"{df_numeric['warping_error'].mean():.6f} ± {df_numeric['warping_error'].std():.6f}",
    # 11. Hausdorff
    "hausdorff_dist"      : "N/A",
    # 12. Line IoU
    "line_iou"            : "N/A",
    # 13. Disconnection Rate
    "disconnection_rate"  : "N/A",
    # 14. Arbitrary t ✅
    "arbitrary_t_support" : arb_t_support,
    # 15. Inference Speed ✅
    "inference_ms"        : f"{df_numeric['inference_ms'].mean():.2f} ms/frame",
    "fps"                 : f"{df_numeric['fps'].mean():.2f} FPS",
    # 16. VRAM Peak ✅
    "vram_peak_mb"        : f"{df_numeric['vram_peak_mb'].max():.1f} MB (peak)",
    # 17. Fine-tuning Speed ⚠️
    "finetune_speed"      : "N/A",
}

# Hiển thị bảng
print("\n" + "=" * 70)
print(f"📊 NORMALIZED 17-METRIC BENCHMARK TABLE — UPR-Net-{MODEL_SIZE}")
print("=" * 70)
for k, v in final_summary.items():
    status = "✅" if v not in ("N/A", None) else "❌"
    print(f"   {status} {k:<25} : {v}")
print("=" * 70)

# Lưu final summary
final_df = pd.DataFrame([final_summary])
final_path = os.path.join(OUTPUT_DIR, "benchmark_17metrics_UPR-Net.csv")
final_df.to_csv(final_path, index=False)
print(f"\n✅ Final 17-metric table saved: {final_path}")

print("\n" + "=" * 70)
print("📁 OUTPUT FILES:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    fsize = os.path.getsize(fpath) / 1024
    print(f"   {fname:45s} {fsize:8.1f} KB")
print("=" * 70)

---
## Cell 9 — Troubleshooting & Common Errors

### 1. CUDA Out of Memory (OOM)
```python
# Giải pháp:
MODEL_SIZE = "base"          # Dùng model nhỏ nhất (~6MB)
torch.cuda.empty_cache()     # Xóa cache GPU
import gc; gc.collect()      # Giải phóng Python memory
```

### 2. BGR/RGB Channel Mismatch
```python
# ❌ Sai:
img = cv2.imread(path)  # BGR — sẽ cho màu sắc sai!

# ✅ Đúng:
img_bgr = cv2.imread(path)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
```

### 3. Padding Error
```python
# UPR-Net yêu cầu H, W chia hết cho:
divisor = 2 ** (pyr_level - 1 + 2)
# pyr_level=3 → divisor=16
# pyr_level=4 → divisor=32

# Công thức padding:
ph = ((h - 1) // divisor + 1) * divisor
pw = ((w - 1) // divisor + 1) * divisor
padding = (0, pw - w, 0, ph - h)   # (left, right, top, bottom)
padded = F.pad(tensor.unsqueeze(0), padding, 'constant', 0.5).squeeze(0)

# Sau inference PHẢI crop lại:
pred = pred[:, :, :h, :w]
```

### 4. ModuleNotFoundError
```python
# Luôn insert path TRƯỚC mọi import:
import sys
sys.path.insert(0, "/kaggle/input/upr-net-source/UPR-Net")  # ← FIRST

from core.pipeline import Pipeline  # ← THEN import
```

### 5. Checkpoint Load Error
```python
# Nếu gặp lỗi CUDA map khi load checkpoint:
state_dict = torch.load(CKPT_PATH, map_location='cpu')  # Load lên CPU trước
# Pipeline tự xử lý move sang GPU
```

### 6. Pyramid Level Quá Thấp/Cao
```python
# Tính tự động theo width:
PYR_LEVEL = math.ceil(math.log2(width / 448) + 3)
# 540p  (960px)  → PYR_LEVEL ≈ 3
# 1080p (1920px) → PYR_LEVEL ≈ 4
# 4K    (3840px) → PYR_LEVEL ≈ 5
```


In [ ]:
# ============================================================
# CELL 9: Verification & Final Check
# ============================================================
print("🔍 Final Verification")
print("=" * 60)

# Đọc lại CSV để verify
df_verify = pd.read_csv(csv_path)
print(f"CSV rows     : {len(df_verify)}")
print(f"CSV columns  : {len(df_verify.columns)}")
print(f"\nColumn list  :")
for col in df_verify.columns:
    sample_val = df_verify[col].iloc[0] if len(df_verify) > 0 else 'N/A'
    print(f"   {col:<25} → {sample_val}")

print("\n" + "=" * 60)
print("📊 Mean metrics over all clips:")
numeric_cols = ["psnr", "ssim", "lpips", "ie_mae", "warping_error",
                "inference_ms", "fps", "vram_peak_mb"]
df_num = df_verify[numeric_cols].apply(pd.to_numeric, errors='coerce')
for col in numeric_cols:
    mean_val = df_num[col].mean()
    print(f"   {col:<25} : {mean_val:.4f}")

print("\n" + "=" * 60)
print("✅ Benchmark hoàn tất!")
print(f"   Tất cả output được lưu tại: {OUTPUT_DIR}")
print("=" * 60)